In [1]:
import pandas as pd
import joblib

pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000) 

In [2]:
import os
import sys

cwd = os.getcwd()
candidates = [
    cwd,
    os.path.join(cwd, "src"),
    os.path.join(cwd, ".."),
    os.path.join(cwd, "..", "src"),
]
for candidate in candidates:
    candidate = os.path.abspath(candidate)
    if candidate not in sys.path:
        sys.path.insert(0, candidate)

from utils.get_k_shortest_paths import get_k_shortest_paths
from utils.get_safest_route_v1 import safest_route
from utils.get_routes_converter import routes_converter


In [3]:
G = joblib.load("../graph/chicago_walk_graph.joblib")

In [4]:
type(G)

networkx.classes.digraph.DiGraph

In [7]:
df = pd.read_csv('../dataset/final_data.csv')

/var/folders/_m/x30w5wkd6l59g090_2t6sv1w0000gn/T/ipykernel_1747/226097962.py:1: DtypeWarning: Columns (0: IUCR, 1: FBI Code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../dataset/final_data.csv')


In [8]:
df['Date'] = pd.to_datetime(df['Date'])
df_sorted_by_latitude_longitude = df.sort_values(by=['Latitude', 'Longitude'])

/var/folders/_m/x30w5wkd6l59g090_2t6sv1w0000gn/T/ipykernel_1747/1582584868.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'])


In [9]:
df_sorted_by_latitude_longitude.head()

,Unnamed: 0,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,Year,Updated On,Latitude,Longitude,Location,Parsed_Date,Hour,Day,Month,DayOfWeek,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos,Lon_R,Lat_R,loc_arrest_ratio_historical,crime_arrest_ratio_historical,loc_domestic_ratio_historical,neighborhood_weighted_crime_index,Crime_Score,IsWeekend,temporal_modifier,CrimeDensity_Community,density_norm,neighborhood_norm,spatial_modifier,risk_score
93136,93136,2008-07-14 22:30:00,004XX E 138TH ST,810,THEFT,OVER $500,STREET,0,0,533,5.0,9.0,54.0,6,2008,02/28/2018 03:56:25 PM,41.644604,-87.610728,"(41.644604096, -87.610728247)",2008-07-14 22:30:00,22,14,7,Monday,-0.500000,0.866025,0.299363,-0.954139,-0.500000,-8.660254e-01,-87.61,41.64,0.0,0.050849,0.0,220.130211,70.0,0,1.448690,428,0.068066,0.047793,0.555902,56.373085
12545,12545,2007-04-14 23:30:00,003XX E 138TH ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,1,1,533,5.0,9.0,54.0,08B,2007,09/07/2021 03:41:02 PM,41.644607,-87.613747,"(41.644606566, -87.613746631)",2007-04-14 23:30:00,23,14,4,Saturday,-0.258819,0.965926,0.299363,-0.954139,0.866025,-5.000000e-01,-87.61,41.64,0.0,0.249455,0.0,218.427269,39.0,0,1.309186,428,0.068066,0.047423,0.555680,28.372063
24714,24714,2020-04-09 22:15:00,003XX E 138TH ST,460,BATTERY,SIMPLE,FACTORY / MANUFACTURING BUILDING,0,0,533,5.0,9.0,54.0,08B,2020,04/17/2020 03:40:23 PM,41.644608,-87.613055,"(41.644607723, -87.613055128)",2020-04-09 22:15:00,22,9,4,Thursday,-0.500000,0.866025,0.968077,-0.250653,0.866025,-5.000000e-01,-87.61,41.64,0.0,0.222486,0.0,204.018125,39.0,0,1.404145,428,0.068066,0.044295,0.553803,30.327195
89258,89258,2008-03-08 00:00:00,003XX E 138TH ST,890,THEFT,FROM BUILDING,FACTORY/MANUFACTURING BUILDING,0,0,533,5.0,9.0,54.0,6,2008,02/28/2018 03:56:25 PM,41.644608,-87.613055,"(41.644607723, -87.613055128)",2008-03-08 00:00:00,0,8,3,Saturday,0.000000,1.000000,0.998717,-0.050649,1.000000,6.123234e-17,-87.61,41.64,0.0,0.058011,0.0,204.018125,70.0,0,1.463072,428,0.068066,0.044295,0.553803,56.717808
93089,93089,2007-08-28 01:30:00,003XX E 138TH ST,810,THEFT,OVER $500,PARKING LOT/GARAGE(NON.RESID.),0,0,533,5.0,9.0,54.0,6,2007,02/10/2018 03:50:01 PM,41.644608,-87.613055,"(41.644607723, -87.613055128)",2007-08-28 01:30:00,1,28,8,Tuesday,0.258819,0.965926,-0.571268,0.820763,-0.866025,-5.000000e-01,-87.61,41.64,0.0,0.050946,0.0,204.018125,70.0,0,1.228570,428,0.068066,0.044295,0.553803,47.627045


In [10]:
lat1, lon1 = df_sorted_by_latitude_longitude['Latitude'].iloc[1], df_sorted_by_latitude_longitude['Longitude'].iloc[0] 
lat2, lon2 = df_sorted_by_latitude_longitude['Latitude'].iloc[9], df_sorted_by_latitude_longitude['Longitude'].iloc[5] 

print('lat 1, lon 1: ', lat1, lon1)
print('lat 2, lon 2: ', lat2, lon2)

lat 1, lon 1:  41.644606566 -87.610728247
lat 2, lon 2:  41.645795847 -87.615192567


In [11]:
routes = get_k_shortest_paths(G, lat1, lon1, lat2, lon2, weight="length", k=10)

In [12]:
routes_converted = routes_converter(routes, G, densify_every_m=300)

Route 0: 5 unique intersection points, cost=282.2m
  -> densified to 5 points (every ~300m along the actual walk path)
Route 1: 6 unique intersection points, cost=292.6m
  -> densified to 6 points (every ~300m along the actual walk path)
Route 2: 8 unique intersection points, cost=310.7m
  -> densified to 8 points (every ~300m along the actual walk path)
Route 3: 9 unique intersection points, cost=321.2m
  -> densified to 9 points (every ~300m along the actual walk path)
Route 4: 10 unique intersection points, cost=326.1m
  -> densified to 10 points (every ~300m along the actual walk path)
Route 5: 9 unique intersection points, cost=327.3m
  -> densified to 9 points (every ~300m along the actual walk path)
Route 6: 11 unique intersection points, cost=337.5m
  -> densified to 11 points (every ~300m along the actual walk path)
Route 7: 12 unique intersection points, cost=347.0m
  -> densified to 12 points (every ~300m along the actual walk path)
Route 8: 11 unique intersection points, co

In [13]:
t_query = pd.Timestamp("2026-07-27 21:00:00")  # 9pm — worth testing at a "riskier" hour

In [15]:
result = safest_route(
     routes=routes_converted, 
     crime_df=df_sorted_by_latitude_longitude, 
     t_query=t_query, 
     bw_space=300, 
     alpha=0.7, 
     beta=0.3, 
     debug=True)

[debug] Temporal_Modifier = 1.413925 (static aggregation, same for every route)
[debug] Spatial rescale bounds: lo=35.0396 hi=64.1795
[debug] route=0 point=0 lat=41.64469 lon=-87.61323 base_severity=39.00 spatial_raw=60.7276 R_i=76.1825
[debug] route=0 point=1 lat=41.64469 lon=-87.61426 base_severity=39.00 spatial_raw=60.7481 R_i=76.2212
[debug] route=0 point=2 lat=41.64469 lon=-87.61435 base_severity=39.00 spatial_raw=60.7532 R_i=76.2310
[debug] route=0 point=3 lat=41.64549 lon=-87.61543 base_severity=40.00 spatial_raw=60.5729 R_i=77.8356
[debug] route=0 point=4 lat=41.64576 lon=-87.61481 base_severity=40.00 spatial_raw=60.4228 R_i=77.5442
[debug] route=1 point=0 lat=41.64469 lon=-87.61323 base_severity=39.00 spatial_raw=60.7276 R_i=76.1825
[debug] route=1 point=1 lat=41.64469 lon=-87.61426 base_severity=39.00 spatial_raw=60.7481 R_i=76.2212
[debug] route=1 point=2 lat=41.64462 lon=-87.61425 base_severity=39.00 spatial_raw=60.7295 R_i=76.1861
[debug] route=1 point=3 lat=41.64469 lon=-

In [16]:
print("\n=== FINAL RESULT ===")
print("Safest route index:", result["safest_route_index"])
print(f"  Mean risk: {result['R_route_mean']:.4f}")
print(f"  Max risk:  {result['R_route_max']:.4f}")
print(f"  Combined:  {result['combined_score']:.4f}")
print("\nAll paths:")
for s in result["all_scores"]:
    print(f"  Route {s['route_index']}:  mean={s['R_route_mean']:.4f}  "
          f"max={s['R_route_max']:.4f}  combined={s['combined_score']:.4f}"
        #   //f"  point_risks={[f'{x:.4f}' for x in s['point_risks']]}"
          )


=== FINAL RESULT ===
Safest route index: 5
  Mean risk: 77.0986
  Max risk:  77.8356
  Combined:  77.3197

All paths:
  Route 0:  mean=77.2401  max=77.8356  combined=77.4188
  Route 1:  mean=77.2029  max=77.8356  combined=77.3927
  Route 2:  mean=77.2199  max=77.8414  combined=77.4063
  Route 3:  mean=77.1866  max=77.8414  combined=77.3831
  Route 4:  mean=77.1699  max=77.8414  combined=77.3714
  Route 5:  mean=77.0986  max=77.8356  combined=77.3197
  Route 6:  mean=77.1379  max=77.8414  combined=77.3489
  Route 7:  mean=77.1946  max=78.2228  combined=77.5031
  Route 8:  mean=77.1274  max=78.2228  combined=77.4560
  Route 9:  mean=77.1636  max=78.2228  combined=77.4814


In [17]:
from utils.visualize_route import visualize_safest_result

visualize_safest_result(result, routes_converted, save_path="safest_route_v1.html")

Saved to safest_route_v1.html
